In [2]:
# %%
import numpy as np

NON1_PATH = "/home/quochuy/Downloads/data_train_fpt/train_ail_data/mini_mfcc_non-speech_1.npy"
NON2_PATH = "/home/quochuy/Downloads/data_train_fpt/train_ail_data/mini_mfcc_non-speech_2.npy"
SPEECH_PATH = "/home/quochuy/Downloads/data_train_fpt/train_ail_data/mini_mfcc_speech.npy"

X_non1 = np.load(NON1_PATH, mmap_mode="r")
X_non2 = np.load(NON2_PATH, mmap_mode="r")
X_speech = np.load(SPEECH_PATH, mmap_mode="r")

print("Non1 :", X_non1.shape)
print("Non2 :", X_non2.shape)
print("Speech:", X_speech.shape)

Non1 : (1094543, 39)
Non2 : (168697, 39)
Speech: (1284608, 39)


In [3]:
# %%
# %%
X_non = np.concatenate([
    X_non1[:, :39],
    X_non2[:, :39]
], axis=0)

X_speech = X_speech[:, :39]

print("Non total:", X_non.shape)
print("Speech:", X_speech.shape)

X = np.vstack([X_non, X_speech]).astype(np.float32)

y = np.hstack([
    np.zeros(len(X_non), dtype=np.int8),
    np.ones(len(X_speech), dtype=np.int8)
])

print("X:", X.shape)
print("Label distribution:", np.bincount(y))

Non total: (1263240, 39)
Speech: (1284608, 39)
X: (2547848, 39)
Label distribution: [1263240 1284608]


In [4]:
# %%
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (2038278, 39)
Test : (509570, 39)


In [5]:
# %%
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)

rf = RandomForestClassifier(
    n_estimators=80,
    max_depth=15,
    min_samples_leaf=10,
    n_jobs=4,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print("=== RANDOM FOREST ===")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))
print("AUC      :", roc_auc_score(y_test, y_prob))

KeyboardInterrupt: 

In [ ]:
# %%
from xgboost import XGBClassifier

xgb = XGBClassifier(
    tree_method="hist",
    device="cuda",          # dùng GPU
    max_depth=10,
    n_estimators=300,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)
y_prob = xgb.predict_proba(X_test)[:, 1]

print("=== XGBOOST (GPU) ===")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))
print("AUC      :", roc_auc_score(y_test, y_prob))

/home/quochuy/miniconda3/envs/audio_ml/lib/python3.10/site-packages/xgboost/core.py:774: UserWarning: [22:29:38] WARNING: /workspace/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


=== XGBOOST (GPU) ===
Accuracy : 0.9866259787664109
Precision: 0.9854806337276337
Recall   : 0.9880313869579094
F1-score : 0.9867543619256417
AUC      : 0.9990309027395399


In [ ]:
# %%
import joblib

# Lưu Random Forest
joblib.dump(rf, "rf_mfcc.pkl")

# Lưu XGBoost
joblib.dump(xgb, "xgb_mfcc.pkl")

print("Models saved successfully!")

Models saved successfully!


In [1]:
# %%
import numpy as np

NON1_PATH = "/home/quochuy/Downloads/data_train_fpt/train_ail_data/mini_stft_non-speech_1.npy"
NON2_PATH = "/home/quochuy/Downloads/data_train_fpt/train_ail_data/mini_stft_non-speech_2.npy"
SPEECH_PATH = "/home/quochuy/Downloads/data_train_fpt/train_ail_data/mini_stft_speech.npy"

MIX_SPEECH_PATH = "/home/quochuy/Downloads/data_train_fpt/train_ail_data/stft/mini_stft_speech_mixed.npy"
MIX_NON_PATH    = "/home/quochuy/Downloads/data_train_fpt/train_ail_data/stft/mini_stft_non-speech_mixed.npy"

# ===== LOAD CLEAN =====
X_non1 = np.load(NON1_PATH, mmap_mode="r")
X_non2 = np.load(NON2_PATH, mmap_mode="r")
X_speech_clean = np.load(SPEECH_PATH, mmap_mode="r")

# ===== LOAD MIXED =====
X_speech_mix = np.load(MIX_SPEECH_PATH, mmap_mode="r")
X_non_mix    = np.load(MIX_NON_PATH, mmap_mode="r")

print("Clean speech:", X_speech_clean.shape)
print("Clean non1 :", X_non1.shape)
print("Clean non2 :", X_non2.shape)
print("Mix speech :", X_speech_mix.shape)
print("Mix non    :", X_non_mix.shape)

# %%
# ===== LẤY 39 FEATURE =====
X_non_clean = np.concatenate([
    X_non1[:, :257],
    X_non2[:, :257]
], axis=0)

X_speech_clean = X_speech_clean[:, :257]
X_speech_mix   = X_speech_mix[:, :257]
X_non_mix      = X_non_mix[:, :257]

print("Clean non total:", X_non_clean.shape)
print("Clean speech   :", X_speech_clean.shape)
print("Mix speech     :", X_speech_mix.shape)
print("Mix non        :", X_non_mix.shape)

# %%
# ===== STACK ALL =====
X = np.vstack([
    X_non_clean,
    X_speech_clean,
    X_non_mix,
    X_speech_mix
]).astype(np.float32)

y = np.hstack([
    np.zeros(len(X_non_clean), dtype=np.int8),
    np.ones(len(X_speech_clean), dtype=np.int8),
    np.zeros(len(X_non_mix), dtype=np.int8),
    np.ones(len(X_speech_mix), dtype=np.int8),
])

print("X:", X.shape)
print("Label distribution:", np.bincount(y))

# %%
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

# %%
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)

from xgboost import XGBClassifier

xgb = XGBClassifier(
    tree_method="hist",
    device="cuda",      # nếu không có GPU thì bỏ dòng này
    max_depth=10,
    n_estimators=300,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)
y_prob = xgb.predict_proba(X_test)[:, 1]

print("=== XGBOOST STFT===")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))
print("AUC      :", roc_auc_score(y_test, y_prob))

# %%

Clean speech: (1284459, 257)
Clean non1 : (1094364, 257)
Clean non2 : (168714, 257)
Mix speech : (849192, 257)
Mix non    : (912072, 257)
Clean non total: (1263078, 257)
Clean speech   : (1284459, 257)
Mix speech     : (849192, 257)
Mix non        : (912072, 257)
X: (4308801, 257)
Label distribution: [2175150 2133651]
Train: (3447040, 257)
Test : (861761, 257)


/home/quochuy/miniconda3/envs/audio_ml/lib/python3.10/site-packages/xgboost/core.py:774: UserWarning: [00:25:00] WARNING: /workspace/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


=== XGBOOST STFT===
Accuracy : 0.9260630267556782
Precision: 0.9397037270315775
Recall   : 0.9090152812896181
F1-score : 0.924104792466225
AUC      : 0.9804248874636032
